In [1]:
import pandas as pd

In [2]:
#  read nav_hitory file

nav = pd.read_csv("../data/raw/nav_history.csv")

print(nav.shape)
nav.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
# Convert date

nav["date"] = pd.to_datetime(nav["date"])
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [5]:
# sort values

nav = nav.sort_values(
    ["amfi_code", "date"]
)
nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [6]:
# Create daily_return column

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change() * 100
)

nav.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-1.030568
5752,100016,2022-01-05,521.7239,1.286515
5753,100016,2022-01-06,515.7880,-1.137747
5754,100016,2022-01-07,515.1639,-0.120999


In [7]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

print("fact_nav updated successfully")

fact_nav updated successfully


In [8]:
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

46000

In [9]:
# Remove duplicates:

nav = nav.drop_duplicates()
print("Shape after removing/drop duplicates:", nav.shape)

Shape after removing/drop duplicates: (46000, 4)


In [10]:
# Validate NAV > 0

print((nav["nav"] <= 0).sum())

0


In [11]:
# clean nav_hstroy file saved in processed folder

nav.to_csv("../data/processed/clean_nav.csv", index=False )

print("Saved")

Saved


In [12]:
# Check NAV values

print("NAV <= 0:", (nav["nav"] <= 0).sum())

print("Duplicate Rows:", nav.duplicated().sum())

print("Shape:", nav.shape)

nav.head()

NAV <= 0: 0
Duplicate Rows: 0
Shape: (46000, 4)


,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-1.030568
5752,100016,2022-01-05,521.7239,1.286515
5753,100016,2022-01-06,515.7880,-1.137747
5754,100016,2022-01-07,515.1639,-0.120999


In [13]:
2. # read investor_transactions file

transactions = pd.read_csv(
    "../data/raw/investor_transactions.csv"
)
transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [14]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [15]:
# Check columns
print(transactions.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [16]:
transactions["amfi_code"].duplicated().sum()

32738

In [17]:
# Standardize format ("Sip -> SIP")

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)

# Convert "Sip" to "SIP"
transactions["transaction_type"] = transactions["transaction_type"].replace({
    "Sip": "SIP"
})

In [18]:
# transaction type

print(transactions["transaction_type"].unique())

['SIP' 'Redemption' 'Lumpsum']


In [19]:
# transaction_date

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [20]:
# Validate amount

print((transactions["amount_inr"] <= 0).sum())

0


In [21]:
# Standardize ("verified" or "pending" -> "Verified and Pending"

transactions["kyc_status"] = (
    transactions["kyc_status"]
    .str.strip()
    .str.title()
)

In [22]:
# Validate KYC (or) checking  kyc status

print( transactions["kyc_status"].value_counts())

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [23]:
# clean investor_transactions file saved in processed folder

transactions.to_csv( "../data/processed/clean_transactions.csv", index=False)
print("saved")

saved


In [24]:
#  read scheme_performance file

performance = pd.read_csv("../data/raw/scheme_performance.csv")
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [25]:
# Negative Sharpe

print(
    (performance["sharpe_ratio"] < 0).sum()
)

0


In [26]:
# Negative Sharpe Ratios
print(
    "Negative Sharpe Ratios:",
    (performance["sharpe_ratio"] < 0).sum()
)

Negative Sharpe Ratios: 0


In [28]:
# Check for missing/non-numeric values:

return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct"
]

for col in return_cols:
   performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )
    

print(performance[return_cols].isnull().sum())

return_1yr_pct       0
return_3yr_pct       0
return_5yr_pct       0
benchmark_3yr_pct    0
dtype: int64


In [29]:
# Flag anomalies -> An anomaly is a value that is unusually high or low.


performance["anomaly_flag"] = (
    (performance["return_1yr_pct"] > 100) |
    (performance["return_1yr_pct"] < -100)
)

print(performance["anomaly_flag"].value_counts())

anomaly_flag
False    40
Name: count, dtype: int64


In [30]:
# Check Expense Ratio Range (0.1% – 2.5%)

performance["expense_ratio_valid"] = (
    performance["expense_ratio_pct"].between(0.1, 2.5)
)

print(performance["expense_ratio_valid"].value_counts())

expense_ratio_valid
True    40
Name: count, dtype: int64


In [31]:
# saved the clean_performance file in processed folder

performance.to_csv("../data/processed/clean_performance.csv",
    index=False
)
print("saved")

saved


In [32]:
# 04 fund_master cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/fund_master.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_fund_master.csv", index=False)

(40, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   fund_house          40 non-null     object 
 2   scheme_name         40 non-null     object 
 3   category            40 non-null     object 
 4   sub_category        40 non-null     object 
 5   plan                40 non-null     object 
 6   launch_date         40 non-null     object 
 7   benchmark           40 non-null     object 
 8   expense_ratio_pct   40 non-null     float64
 9   exit_load_pct       40 non-null     float64
 10  min_sip_amount      40 non-null     int64  
 11  min_lumpsum_amount  40 non-null     int64  
 12  fund_manager        40 non-null     object 
 13  risk_category       40 non-null     object 
 14  sebi_category_code  40 non-null     object 
dtypes: float64(2), int64(3), object(10)
memory usage: 

In [33]:
# 05 aum_by_fund_house cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/aum_by_fund_house.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_aum_by_fund_house.csv", index=False)

(90, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            90 non-null     object 
 1   fund_house      90 non-null     object 
 2   aum_lakh_crore  90 non-null     float64
 3   aum_crore       90 non-null     int64  
 4   num_schemes     90 non-null     int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 3.6+ KB
None
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64


In [34]:
# print(df.info())
df.groupby("date")["aum_lakh_crore"].sum()

date
2022-03-31    30.18
2022-09-30    30.90
2023-03-31    32.35
2023-09-30    37.42
2024-03-31    43.75
2024-09-30    47.27
2024-12-31    52.68
2025-03-31    54.47
2025-12-31    62.74
Name: aum_lakh_crore, dtype: float64

In [35]:
# 05 monthly_sip_inflows cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/monthly_sip_inflows.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

df["yoy_growth_pct"] = df["yoy_growth_pct"].fillna(0)

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_monthly_sip_inflows.csv", index=False)

(48, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   month                      48 non-null     object 
 1   sip_inflow_crore           48 non-null     int64  
 2   active_sip_accounts_crore  48 non-null     float64
 3   new_sip_accounts_lakh      48 non-null     float64
 4   sip_aum_lakh_crore         48 non-null     float64
 5   yoy_growth_pct             36 non-null     float64
dtypes: float64(4), int64(1), object(1)
memory usage: 2.4+ KB
None
month                        0
sip_inflow_crore             0
active_sip_accounts_crore    0
new_sip_accounts_lakh        0
sip_aum_lakh_crore           0
yoy_growth_pct               0
dtype: int64


In [42]:
# 06 portfolio_holdings cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/portfolio_holdings.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_portfolio_holdings.csv", index=False)

(322, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   amfi_code          322 non-null    int64  
 1   stock_symbol       322 non-null    object 
 2   stock_name         322 non-null    object 
 3   sector             322 non-null    object 
 4   weight_pct         322 non-null    float64
 5   market_value_cr    322 non-null    float64
 6   current_price_inr  322 non-null    float64
 7   portfolio_date     322 non-null    object 
dtypes: float64(3), int64(1), object(4)
memory usage: 20.3+ KB
None
amfi_code            0
stock_symbol         0
stock_name           0
sector               0
weight_pct           0
market_value_cr      0
current_price_inr    0
portfolio_date       0
dtype: int64


In [36]:
# 04 benchmark_indices cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/benchmark_indices.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_benchmark_indices.csv", index=False)

(8050, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         8050 non-null   object 
 1   index_name   8050 non-null   object 
 2   close_value  8050 non-null   float64
dtypes: float64(1), object(2)
memory usage: 188.8+ KB
None
date           0
index_name     0
close_value    0
dtype: int64


In [5]:
# 04 category_inflows cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/category_inflows.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_category_inflows.csv", index=False)

(144, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   month             144 non-null    object
 1   category          144 non-null    object
 2   net_inflow_crore  144 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 3.5+ KB
None
month               0
category            0
net_inflow_crore    0
dtype: int64


In [37]:
# 04 industry_folio_count cleaning the file

import pandas as pd

df = pd.read_csv("../data/raw/industry_folio_count.csv")

# Check dataset
print(df.shape)
print(df.info())

# Remove duplicates
df = df.drop_duplicates()

# Remove rows that are completely empty
df = df.dropna(how="all")

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown").str.strip()

# Convert numeric columns
for col in df.select_dtypes(include="object").columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print(df.isnull().sum())

# Save cleaned file
df.to_csv("../data/processed/clean_industry_folio_count.csv", index=False)

(21, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                21 non-null     object 
 1   total_folios_crore   21 non-null     float64
 2   equity_folios_crore  21 non-null     float64
 3   debt_folios_crore    21 non-null     float64
 4   hybrid_folios_crore  21 non-null     float64
 5   others_folios_crore  21 non-null     float64
dtypes: float64(5), object(1)
memory usage: 1.1+ KB
None
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64


In [38]:
import pandas as pd

portfolio = pd.read_csv("../data/processed/clean_portfolio_holdings.csv")
portfolio.head()

,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31


In [39]:
portfolio.to_sql(
    "fact_portfolio",
    engine,
    if_exists="append",
    index=False
)

322

In [40]:
import pandas as pd

df = pd.read_csv("../data/processed/clean_aum_by_fund_house.csv")
print(df.columns)


Index(['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes'], dtype='object')


In [41]:
import os

print(os.path.exists("../data/processed/clean_aum_by_fund_house.csv"))

True
